# Lesson 06: Building MCP Clients & Autonomous Gemini LLM Hosts

In this notebook, you will:
1. Connect an in-memory MCP Client session to a `FastMCP` server.
2. Translate MCP `inputSchema` definitions into `google-genai` `FunctionDeclaration` objects.
3. Run an autonomous `gemini-2.5-flash` tool-calling loop with graceful offline fallback.

In [ ]:
!pip install -q -U mcp google-genai python-dotenv

In [ ]:
import os
import math
from mcp.server.fastmcp import FastMCP
from mcp.shared.memory import create_connected_server_and_client_session
from google import genai
from google.genai import types

def get_gemini_client():
    api_key = None
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    api_key = api_key or os.environ.get("GEMINI_API_KEY")
    return genai.Client(api_key=api_key) if api_key else None

mcp = FastMCP("HostDemoServer")

@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

@mcp.tool()
def sqrt(x: float) -> float:
    """Square root of a non-negative number."""
    return math.sqrt(x)

async def run_host(query: str):
    async with create_connected_server_and_client_session(mcp._mcp_server) as session:
        tools_resp = await session.list_tools()
        print("Discovered MCP tools:", [t.name for t in tools_resp.tools])

        client = get_gemini_client()
        if not client:
            res1 = await session.call_tool("multiply", {"a": 24, "b": 15})
            print("[Offline Demo] Direct MCP call multiply(24, 15) ->", res1.content[0].text)
            return

        decls = [
            types.FunctionDeclaration(name=t.name, description=t.description or "", parameters=t.inputSchema)
            for t in tools_resp.tools
        ]
        resp = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=query,
            config=types.GenerateContentConfig(tools=[types.Tool(function_declarations=decls)]),
        )
        if resp.function_calls:
            for fc in resp.function_calls:
                tool_out = await session.call_tool(fc.name, dict(fc.args))
                print(f"Gemini called {fc.name}({dict(fc.args)}) -> {tool_out.content[0].text}")
        else:
            print(resp.text)

await run_host("What is 24 multiplied by 15?")

### Challenge Exercise
Extend `run_host()` into a `while` loop that sends `types.Part.from_function_response(...)` back to `gemini-2.5-flash` so the model can chain `multiply(24, 15)` and `sqrt(144)` in a single user conversation.